<a href="https://colab.research.google.com/github/datweb07/repo-nosql/blob/main/baitap1_nosql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.0 MB/s eta 0:00:00


In [2]:
from datetime import datetime
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from google.colab import userdata

uri = userdata.get('MONGODB_URI')

client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Kết nối MongoDB thành công!")
except Exception as e:
    print(e)

Kết nối MongoDB thành công!


In [4]:
db = client['QL_SV_PHIM']

collection_sv = db['sinhvien']
collection_phim = db['phim']
print("đã tạo thành công db và collection!\n")

đã tạo thành công db và collection!



In [ ]:
sinhvien = [
    {
        "masv": "SV001",
        "hoten": "Nguyen Van An",
        "dtb": 7.0,
        "drl": 8.5,
        "chuyennganh": {"manganh": "IT", "tennganh": "Cong nghe thong tin"},
        "dchuyennganh": [
            {"tenmon": "Toan roi rac", "diem": 8.0},
            {"tenmon": "Lap trinh Python", "diem": 9.0}
        ],
        "dienthoai": ["0901234567", "0987654321"],
        "nguoithan": [
            {"hoten": "Nguyen Van B", "namsinh": 1970, "quanhe": "Cha"}
        ]
    },
    {
        "masv": "SV002",
        "hoten": "Le Thi Binh",
        "dtb": 8.0,
        "drl": 6.5,
        "chuyennganh": {"manganh": "KT", "tennganh": "Ke toan"},
        "dchuyennganh": [
            {"tenmon": "Toan cao cap", "diem": 4.0},
            {"tenmon": "Nguyen ly ke toan", "diem": 8.0}
        ],
        "dienthoai": ["0911222333"],
        "nguoithan": []
    },
    {
        "masv": "SV003",
        "hoten": "Tran Van Cuong",
        "dtb": 6.5,
        "drl": 4.0,
        "chuyennganh": {"manganh": "MKT", "tennganh": "Marketing"},
        "dchuyennganh": [
            {"tenmon": "Marketing can ban", "diem": 7.0}
        ],
        "dienthoai": [],
        "nguoithan": [
            {"hoten": "Tran Van D", "namsinh": 1975, "quanhe": "Cha"},
            {"hoten": "Nguyen Thi E", "namsinh": 1978, "quanhe": "Me"}
        ]
    }
]

phim = [
    {
        "maphim": "P001",
        "tenphim": "Tinh Yeu Va Noi So",
        "thoigian": 120,
        "tuoitoithieu": 13,
        "theloai": ["kinh di", "tinh cam", "hai"],
        "daodien": "Victor Vu",
        "dienvien": [
            {"ten": "Thai Hoa", "tuoi": 40, "theloaidien": ["kinh di", "hai"]},
            {"ten": "Kieu Minh Tuan", "tuoi": 35, "theloaidien": ["hai"]}
        ],
        "ctychieu": [
            {
                "macty": "LOT",
                "SoLuongSuat": 150,
                "TongSoVe": 12000,
                "ngbd": datetime(2024, 1, 1),
                "ngkt": datetime(2024, 1, 30)
            },
            {
                "macty": "CGV",
                "SoLuongSuat": 200,
                "TongSoVe": 15000,
                "ngbd": datetime(2024, 1, 5),
                "ngkt": datetime(2024, 2, 5)
            }
        ]
    },
    {
        "maphim": "P002",
        "tenphim": "Cuoc Chien Ngam",
        "thoigian": 90,
        "tuoitoithieu": 16,
        "theloai": ["hanh dong", "hinh su"],
        "daodien": "Charlie Nguyen",
        "dienvien": [
            {"ten": "Johnny Tri Nguyen", "tuoi": 45, "theloaidien": ["hanh dong"]},
            {"ten": "Ngo Thanh Van", "tuoi": 40, "theloaidien": ["hanh dong"]},
            {"ten": "Dustin Nguyen", "tuoi": 50, "theloaidien": ["hanh dong"]}
        ],
        "ctychieu": [
            {
                "macty": "BHD",
                "SoLuongSuat": 80,
                "TongSoVe": 5000,
                "ngbd": datetime(2024, 3, 1),
                "ngkt": datetime(2024, 3, 15)
            }
        ]
    }
]

collection_sv.insert_many(sinhvien)

collection_phim.insert_many(phim)

print("tạo dữ liệu thành công!")

tạo dữ liệu thành công!


# Danh sách các phim mà có thời gian chiếu nhỏ hơn 100 hoặc là những phim mà có hơn 2 diễn viên đóng

In [5]:
q1 = {
    "$or": [
        {
            "thoigian": {"$lt": 100}
        },
        {
            "$expr": {
                "$gt": [
                    {
                        "$size": "$dienvien"
                    },
                    2
                ]
            }
        }
    ]
}

res_q1 = collection_phim.find(q1, {
    "_id": 0,
    "maphim": 1,
    "tenphim": 1,
})
for i in res_q1:
    print(i)

{'maphim': 'P002', 'tenphim': 'Cuoc Chien Ngam'}


# Danh sách các phim mà có công ty chiếu tên là `LOT` và số vé của công ty đó là hơn 10000 vé.

In [6]:
q2 = collection_phim.find(
    {
        "ctychieu": {
            "$elemMatch": {
                "macty": "LOT",
                "TongSoVe": {"$gt": 10000}
          }
      }
    },
    {
        "_id": 0,
        "maphim": 1,
        "tenphim": 1,
    }
)

for i in q2:
  print(i)

{'maphim': 'P001', 'tenphim': 'Tinh Yeu Va Noi So'}


# Danh sách các phim mà dành cho tuổi tối thiểu là 13 nhưng phim đó lại được xếp đủ cả 2 thể loại `kinh di` và `tinh cam`

In [ ]:
q3 = collection_phim.find(
    {
        "tuoitoithieu": 13,
        "theloai": {
            "$all": ["kinh di", "tinh cam"]
        }
    },
    {
        "_id": 0,
        "maphim": 1,
        "tenphim": 1,
    }
)

for i in q3:
  print(i)

{'maphim': 'P001', 'tenphim': 'Tinh Yeu Va Noi So'}


# Danh sách các sinh viên mà trung bình điểm các môn học chuyên ngành lớn hơn điểm trung bình của sinh viên đó

In [7]:
q4 = collection_sv.aggregate([
    {
        "$match": {
            "$expr": {
                "$gt": [
                    {
                        "$avg": "$dchuyennganh.diem"
                    },
                    "$dtb"
                ]
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "masv": 1,
            "hoten": 1,
            "dtb": 1,
            "dchuyennganh": 1
        }
    }
]
)

for i in q4:
  print(i)

{'masv': 'SV001', 'hoten': 'Nguyen Van An', 'dtb': 7.0, 'dchuyennganh': [{'tenmon': 'Toan roi rac', 'diem': 8.0}, {'tenmon': 'Lap trinh Python', 'diem': 9.0}]}
{'masv': 'SV003', 'hoten': 'Tran Van Cuong', 'dtb': 6.5, 'dchuyennganh': [{'tenmon': 'Marketing can ban', 'diem': 7.0}]}


# Danh sách các sinh viên, cột dtb_cuoi_cung là dtb cộng thêm 1 điểm nếu điểm rèn luyện lớn hơn bằng  5 và nhỏ hơn 8, và cộng thêm 2 điểm nếu điểm rèn luyện lớn hơn bằng 8, trường hợp điểm rèn luyện nhỏ hơn 5 thì không cộng

In [ ]:
q5 = [
    {
        "$addFields": {
            "dtb_cuoi_cung": {
                "$add": [
                    "$dtb",
                    {
                        "$switch": {
                            "branches": [
                                #case 1: drl >= 8 -> cộng 2
                                {"case": {"$gte": ["$drl", 8]}, "then": 2},
                                #case 2: 5 <= drl < 8 -> cộng 1
                                {"case": {"$and": [{"$gte": ["$drl", 5]}, {"$lt": ["$drl", 8]}]}, "then": 1},
                            ],
                            "default": 0
                        }
                    }
                ]
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "masv": 1,
            "hoten": 1,
            "dtb": 1,
            "drl": 1,
            "dtb_cuoi_cung": {
                "$round": ["$dtb_cuoi_cung", 1]
            }
        }
    }
]

res_5 = collection_sv.aggregate(q5)
for i in res_5:
  print(i)

{'masv': 'SV001', 'hoten': 'Nguyen Van An', 'dtb': 7.0, 'drl': 8.5, 'dtb_cuoi_cung': 9.0}
{'masv': 'SV002', 'hoten': 'Le Thi Binh', 'dtb': 8.0, 'drl': 6.5, 'dtb_cuoi_cung': 9.0}
{'masv': 'SV003', 'hoten': 'Tran Van Cuong', 'dtb': 6.5, 'drl': 4.0, 'dtb_cuoi_cung': 6.5}


# Danh sách các sinh viên và môn chuyên ngành (mà là môn có điểm là cao nhất trong các điểm của các môn chuyên ngành của sinh viên đó).

In [11]:
q6 = [
    {
        "$unwind": "$dchuyennganh"
    },
    {
        "$sort":
         {
             "dchuyennganh.diem": -1        #sắp xếp giảm dần
         }
    },
    {
        "$group": {
            "_id": "$_id",
            "masv": {
                "$first": "$masv"
            },
            "hoten": {
                "$first": "$hoten"
            },
            "max": {
                "$first": "$dchuyennganh"
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "masv": 1,
            "hoten": 1,
            "diem_cao_nhat": "$max.diem",
            "mon_hoc": "$max.tenmon"
        }
    }
]

res_q6 = collection_sv.aggregate(q6)
for i in res_q6:
  print(i)

{'masv': 'SV001', 'hoten': 'Nguyen Van An', 'diem_cao_nhat': 9.0, 'mon_hoc': 'Lap trinh Python'}
{'masv': 'SV002', 'hoten': 'Le Thi Binh', 'diem_cao_nhat': 8.0, 'mon_hoc': 'Nguyen ly ke toan'}
{'masv': 'SV003', 'hoten': 'Tran Van Cuong', 'diem_cao_nhat': 7.0, 'mon_hoc': 'Marketing can ban'}


# Danh sách các sinh viên (mã sv, họ tên, điểm chuyên ngành) mà mọi điểm chuyên ngành đều lớn hơn hay bằng 5

In [ ]:
q7 = collection_sv.find(
    {
        "dchuyennganh": {
            "$not": {
                "$elemMatch": {
                    "diem": {"$lt": 5}
                }
            }
        }
    },
    {
        "_id": 0,
        "masv": 1,
        "hoten": 1,
    }
)

for i in q7:
  print(i)

{'masv': 'SV001', 'hoten': 'Nguyen Van An'}
{'masv': 'SV003', 'hoten': 'Tran Van Cuong'}
